# Virtual Try-On v4 — HR-VITON Pretrained Inference

**HR-VITON (ECCV 2022)** — High-Resolution Virtual Try-On with Misalignment and Occlusion-Handled Conditions
*Lee et al., 2022 · [Paper](https://arxiv.org/abs/2206.14180) · [Code](https://github.com/sangyun884/HR-VITON)*

This notebook runs the **official pretrained HR-VITON models** end-to-end:

| Stage | Module | Role |
|---|---|---|
| 1 | **ConditionGenerator (tocg)** | Jointly warps cloth + predicts segmentation (5-scale flow) |
| 2 | **SPADEGenerator** | Synthesises final try-on image with ALIAS normalisation |

Dataset: VITON-HD (768×1024)
Expected SSIM ≥ 0.80 (paired), PSNR ≥ 25 dB


## 1. Install Dependencies

In [ ]:
%%capture
import subprocess, sys

pkgs = [
    "gdown",           # Google Drive download
    "kornia",          # replaces torchgeometry GaussianBlur
    "tensorboardX",    # imported by HR-VITON networks
    "scikit-image",    # SSIM / PSNR metrics
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✓ Dependencies installed")


## 2. Download & Extract Dataset

In [ ]:
import zipfile, subprocess
from pathlib import Path

DATASET_DIR = Path('./dataset')
ZIP_PATH    = DATASET_DIR / 'complete dataset.zip'
KAGGLE_URL  = 'https://www.kaggle.com/api/v1/datasets/download/marquis03/high-resolution-viton-zalando-dataset'

def dataset_ready():
    p = DATASET_DIR / 'test' / 'image'
    return p.exists() and len(list(p.iterdir())) > 100

if dataset_ready():
    print('Dataset already present.')
else:
    DATASET_DIR.mkdir(exist_ok=True)
    if not ZIP_PATH.exists():
        print('Downloading (~4.4 GB) ...')
        subprocess.run(['curl', '-L', '-o', str(ZIP_PATH), KAGGLE_URL], check=True)
    print('Extracting ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DATASET_DIR)
    print('Done.')

print(f'Train images: {len(list((DATASET_DIR/"train"/"image").iterdir())):,}')
print(f'Test  images: {len(list((DATASET_DIR/"test" /"image").iterdir())):,}')


## 3. Locate Dataset Paths

In [ ]:
import os, glob
from pathlib import Path

# Known structure after download: ./dataset/train/ and ./dataset/test/
DATASET_DIR = Path('./dataset')

DATA_ROOT  = str(DATASET_DIR)          # opt.dataroot
TEST_SPLIT = str(DATASET_DIR / 'test') # actual data lives here

# Find the pairs file (test_pairs.txt)
pairs_file = None
for candidate in [
    DATASET_DIR / 'test_pairs.txt',
    DATASET_DIR / 'test' / 'test_pairs.txt',
]:
    if candidate.is_file():
        pairs_file = str(candidate)
        break

if pairs_file is None:
    found = glob.glob(str(DATASET_DIR / '**' / '*pairs*.txt'), recursive=True)
    if found:
        pairs_file = sorted(found)[0]

assert pairs_file, "test_pairs.txt not found — check dataset structure"

with open(pairs_file) as f:
    lines = f.readlines()

print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"Pairs file : {pairs_file}")
print(f"Test pairs : {len(lines)}")
print("First 3 pairs:")
for l in lines[:3]:
    print(" ", l.strip())


## 4. Exploratory Data Analysis

We analyse the **VITON-HD test split** across all modalities before running inference.
Seven subsections cover dataset composition, image statistics, semantic parse maps,
pose keypoints, colour distributions, and cloth-mask geometry.


### 4.1 Dataset Overview

In [ ]:
import os, json, random
from pathlib import Path
from PIL import Image
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.ndimage import gaussian_filter

# Reuse paths from §3
TEST_DIR = DATASET_DIR / "test"

def load_pairs(filepath):
    pairs = []
    with open(filepath) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                pairs.append((parts[0], parts[1]))
    return pairs

test_pairs = load_pairs(pairs_file)
print(f"Test pairs : {len(test_pairs):,}")
print(f"Sample     : {test_pairs[0]}")

modalities = sorted([d for d in os.listdir(TEST_DIR) if (TEST_DIR / d).is_dir()])
print(f"\nModalities ({len(modalities)}):")
for mod in modalities:
    count = len(list((TEST_DIR / mod).iterdir()))
    print(f"  {mod:<45s}: {count:,} files")


### 4.2 Sample Pair Visualisation
Five modalities shown for three test pairs: person image, garment, cloth mask, semantic parse map, and agnostic image (body with clothing region blanked).

In [ ]:
PARSE_COLORS = np.array([
    [0,0,0],[128,0,0],[255,0,0],[0,85,0],[170,0,51],
    [255,85,0],[0,0,85],[0,119,221],[85,85,0],[0,85,85],
    [85,51,0],[52,86,128],[0,128,0],[0,0,255],[51,170,221],
    [0,255,255],[85,255,170],[170,255,85],
], dtype=np.uint8)

LABEL_NAMES = {
    0:'Background',1:'Hat',2:'Hair',3:'Sunglasses',4:'Upper-cloth',
    5:'Dress/Upper',6:'Coat',7:'Socks',8:'Pants',9:'Right Shoe',
    10:'Left Shoe',11:'Face',12:'Left Leg',13:'Right Leg',
    14:'Left Arm',15:'Right Arm',16:'Bag',17:'Scarf',
}

def colorize_parse(parse_np):
    H, W = parse_np.shape
    rgb  = np.zeros((H, W, 3), dtype=np.uint8)
    for lbl in np.unique(parse_np):
        if lbl < len(PARSE_COLORS):
            rgb[parse_np == lbl] = PARSE_COLORS[lbl]
    return rgb

sample_idx = [0, len(test_pairs)//3, 2*len(test_pairs)//3]
fig, axes  = plt.subplots(3, 5, figsize=(20, 13))
fig.suptitle("Sample Test Pairs — Five Modalities", fontsize=15, fontweight='bold', y=1.01)

col_titles = ["Person Image", "Garment", "Cloth Mask", "Parse Map (colour)", "Agnostic Image"]
for c, t in enumerate(col_titles):
    axes[0][c].set_title(t, fontsize=11, fontweight='bold')

for row, idx in enumerate(sample_idx):
    p_name, c_name = test_pairs[idx]
    base = p_name.replace('.jpg', '')
    person_np   = np.array(Image.open(TEST_DIR / "image"          / p_name))
    cloth_np    = np.array(Image.open(TEST_DIR / "cloth"          / c_name))
    mask_np     = np.array(Image.open(TEST_DIR / "cloth-mask"     / c_name).convert('L'))
    parse_np    = np.array(Image.open(TEST_DIR / "image-parse-v3" / (base + ".png")))
    agnostic_np = np.array(Image.open(TEST_DIR / "agnostic-v3.2"  / p_name))

    for col, (img, cm) in enumerate(zip(
        [person_np, cloth_np, mask_np, colorize_parse(parse_np), agnostic_np],
        [None, None, 'gray', None, None]
    )):
        axes[row][col].imshow(img, cmap=cm)
        axes[row][col].axis('off')
    axes[row][0].set_ylabel(f"Pair {row+1}", fontsize=10, rotation=90, labelpad=10)

plt.tight_layout()
plt.savefig("eda_sample_pairs.png", dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved → eda_sample_pairs.png")


### 4.3 Image Statistics
Per-channel mean pixel values and brightness distributions across 50 random test samples.

In [ ]:
N_SAMPLE = 50
rng_pairs = random.sample(test_pairs, N_SAMPLE)

person_means, cloth_means = [], []
person_bright, cloth_bright = [], []

for p_name, c_name in rng_pairs:
    p_arr = np.array(Image.open(TEST_DIR / "image" / p_name)).astype(np.float32)
    c_arr = np.array(Image.open(TEST_DIR / "cloth" / c_name)).astype(np.float32)
    person_means.append(p_arr.mean(axis=(0,1)))
    cloth_means.append(c_arr.mean(axis=(0,1)))
    person_bright.append(p_arr.mean())
    cloth_bright.append(c_arr.mean())

person_means = np.array(person_means)
cloth_means  = np.array(cloth_means)

sample_img = np.array(Image.open(TEST_DIR / "image" / test_pairs[0][0]))
print(f"Image shape : {sample_img.shape}  (H × W × C)")
print(f"Dtype       : {sample_img.dtype}")
print(f"\nChannel-wise mean ({N_SAMPLE} samples):")
print(f"  {'Ch':>4}  {'Person':>10}  {'Garment':>10}")
for i, ch in enumerate(['R','G','B']):
    print(f"  {ch:>4}  {person_means[:,i].mean():10.2f}  {cloth_means[:,i].mean():10.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Image Statistics — 50 Random Test Samples", fontsize=14, fontweight='bold')

sns.histplot(person_bright, bins=20, color='steelblue',  label='Person',  kde=True, ax=axes[0], alpha=0.6)
sns.histplot(cloth_bright,  bins=20, color='darkorange', label='Garment', kde=True, ax=axes[0], alpha=0.6)
axes[0].set_xlabel("Mean pixel brightness (0–255)"); axes[0].set_ylabel("Count")
axes[0].set_title("Brightness Distribution"); axes[0].legend(); axes[0].grid(alpha=0.3)

x, w = np.arange(3), 0.35
axes[1].bar(x - w/2, person_means.mean(0), w, label='Person',  color='steelblue',  alpha=0.8)
axes[1].bar(x + w/2, cloth_means.mean(0),  w, label='Garment', color='darkorange', alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(['R','G','B'])
axes[1].set_ylabel("Mean pixel value"); axes[1].set_title("Per-Channel Mean")
axes[1].legend(); axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("eda_image_stats.png", dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved → eda_image_stats.png")


### 4.4 Parse-Map Label Analysis
Label frequency (% of pixels) across 50 test images. The upper-clothes label (5) dominates the clothing region.

In [ ]:
N_SAMPLE = 50
sample_pairs = random.sample(test_pairs, N_SAMPLE)
label_counts = Counter()
total_pixels = 0

for p_name, _ in sample_pairs:
    base = p_name.replace('.jpg', '')
    parse_arr = np.array(Image.open(TEST_DIR / "image-parse-v3" / (base + ".png")))
    label_counts.update(parse_arr.flatten().tolist())
    total_pixels += parse_arr.size

sorted_labels = sorted(label_counts.keys())
label_pct = {l: label_counts[l] / total_pixels * 100 for l in sorted_labels}
label_str = [LABEL_NAMES.get(l, f"Label {l}") for l in sorted_labels]

print("Label frequencies (% pixels, 50 images):")
for lbl, name, pct in zip(sorted_labels, label_str, [label_pct[l] for l in sorted_labels]):
    print(f"  {lbl:3d}  {name:<18s}: {pct:6.2f}%  {'#' * int(pct)}")

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
fig.suptitle("Parse-Map Label Analysis (50 test images)", fontsize=14, fontweight='bold')

hex_colors = [
    f"#{PARSE_COLORS[l][0]:02x}{PARSE_COLORS[l][1]:02x}{PARSE_COLORS[l][2]:02x}"
    if l < len(PARSE_COLORS) else '#888888'
    for l in sorted_labels
]
bars = axes[0].bar(label_str, [label_pct[l] for l in sorted_labels],
                   color=hex_colors, edgecolor='black', linewidth=0.5)
axes[0].set_xticklabels(label_str, rotation=45, ha='right', fontsize=9)
axes[0].set_ylabel("% of total pixels"); axes[0].set_title("Label Frequency")
axes[0].grid(alpha=0.3, axis='y')
for bar, pct in zip(bars, [label_pct[l] for l in sorted_labels]):
    if pct > 1.0:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                     f"{pct:.1f}%", ha='center', va='bottom', fontsize=7)

axes[1].axis('off')
sub = gridspec.GridSpecFromSubplotSpec(2, 3, subplot_spec=axes[1].get_subplotspec(),
                                       wspace=0.05, hspace=0.05)
for i in range(6):
    p_name, _ = sample_pairs[i]
    base = p_name.replace('.jpg', '')
    parse_arr = np.array(Image.open(TEST_DIR / "image-parse-v3" / (base + ".png")))
    ax_sub = fig.add_subplot(sub[i//3, i%3])
    ax_sub.imshow(colorize_parse(parse_arr)); ax_sub.axis('off')
    ax_sub.set_title(f"#{i+1}", fontsize=8)

plt.tight_layout()
plt.savefig("eda_parse_map.png", dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved → eda_parse_map.png")


### 4.5 OpenPose Keypoint Analysis
Spatial heatmap and confidence distributions for 11 upper-body keypoints (BODY_25 format).

In [ ]:
KEYPOINT_NAMES = {
    0:'Nose', 1:'Neck', 2:'RShoulder', 3:'RElbow', 4:'RWrist',
    5:'LShoulder', 6:'LElbow', 7:'LWrist', 8:'MidHip',
    9:'RHip', 10:'RKnee', 11:'RAnkle', 12:'LHip', 13:'LKnee', 14:'LAnkle',
    15:'REye', 16:'LEye', 17:'REar', 18:'LEar',
    19:'LBigToe', 20:'LSmallToe', 21:'LHeel', 22:'RBigToe', 23:'RSmallToe', 24:'RHeel',
}
UPPER_BODY_IDX = [0,1,2,3,4,5,6,7,8,9,12]

N_SAMPLE  = 50
json_dir  = TEST_DIR / "openpose_json"
json_files = sorted(json_dir.iterdir())[:N_SAMPLE]

conf_matrix = np.zeros((N_SAMPLE, 25))
heatmap_sum = np.zeros((256, 192))

for fi, jpath in enumerate(json_files):
    try:
        with open(jpath) as f:
            data = json.load(f)
        people = data.get('people', [])
        if not people: continue
        kpts = np.array(people[0].get('pose_keypoints_2d', [0]*75)).reshape(25, 3)
        conf_matrix[fi] = kpts[:, 2]
        for k in range(25):
            x, y, c = kpts[k]
            if c > 0.1:
                xi = int(np.clip(x / 768 * 192, 0, 191))
                yi = int(np.clip(y / 1024 * 256, 0, 255))
                heatmap_sum[yi, xi] += c
    except Exception:
        pass

heatmap_smooth = gaussian_filter(heatmap_sum, sigma=6)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("OpenPose Keypoint Analysis (50 test images)", fontsize=14, fontweight='bold')

im = axes[0].imshow(heatmap_smooth, cmap='hot', origin='upper', aspect='auto')
axes[0].set_title("Keypoint Position Heatmap (confidence-weighted)")
axes[0].set_xlabel("x (→192 px)"); axes[0].set_ylabel("y (→256 px)")
plt.colorbar(im, ax=axes[0], label='Accumulated confidence')

conf_data  = [conf_matrix[:, k] for k in UPPER_BODY_IDX]
kpt_labels = [KEYPOINT_NAMES[k] for k in UPPER_BODY_IDX]
bp = axes[1].boxplot(conf_data, labels=kpt_labels, patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], plt.cm.Set2(np.linspace(0,1,len(UPPER_BODY_IDX)))):
    patch.set_facecolor(color)
axes[1].set_xticklabels(kpt_labels, rotation=45, ha='right', fontsize=9)
axes[1].set_ylabel("Confidence [0–1]"); axes[1].set_title("Upper-Body Keypoint Confidence")
axes[1].axhline(0.5, color='red', linestyle='--', linewidth=1, label='Threshold=0.5')
axes[1].legend(); axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("eda_keypoints.png", dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean upper-body confidence : {conf_matrix[:, UPPER_BODY_IDX].mean():.3f}")
print(f"Fraction > 0.5             : {(conf_matrix[:, UPPER_BODY_IDX] > 0.5).mean()*100:.1f}%")
print("✓ Saved → eda_keypoints.png")


### 4.6 Colour Distribution Analysis
KDE plots per RGB channel comparing person images vs garment images.

In [ ]:
N_SAMPLE = 50
sample_p = random.sample(test_pairs, N_SAMPLE)

person_channels = [[], [], []]
cloth_channels  = [[], [], []]

for p_name, c_name in sample_p:
    p_arr = np.array(Image.open(TEST_DIR / "image" / p_name))
    c_arr = np.array(Image.open(TEST_DIR / "cloth" / c_name))
    for ch in range(3):
        person_channels[ch].extend(p_arr.reshape(-1,3)[::50, ch].tolist())
        cloth_channels[ch].extend(c_arr.reshape(-1,3)[::50, ch].tolist())

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("RGB Channel Distributions — Person vs. Garment (50 test samples)",
             fontsize=14, fontweight='bold')

for i, (name, color) in enumerate(zip(['Red','Green','Blue'], ['#e74c3c','#27ae60','#2980b9'])):
    ax = axes[i]
    sns.kdeplot(person_channels[i], ax=ax, color=color, label='Person',
                linewidth=2, fill=True, alpha=0.3)
    sns.kdeplot(cloth_channels[i],  ax=ax, color=color, label='Garment',
                linewidth=2, fill=True, alpha=0.15, linestyle='--')
    ax.set_xlabel(f"{name} channel (0–255)"); ax.set_ylabel("Density")
    ax.set_title(f"{name} Channel"); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("eda_color_dist.png", dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved → eda_color_dist.png")


### 4.7 Cloth-Mask Coverage Analysis
Fraction of image area covered by the garment mask and bounding-box aspect ratio (200 samples).

In [ ]:
N_SAMPLE = 200
sample_p = random.sample(test_pairs, min(N_SAMPLE, len(test_pairs)))

coverages, aspect_ratios = [], []

for _, c_name in sample_p:
    mask_path = TEST_DIR / "cloth-mask" / c_name
    if not mask_path.exists(): continue
    mask   = np.array(Image.open(mask_path).convert('L'))
    H, W   = mask.shape
    binary = mask > 127
    coverages.append(binary.sum() / (H * W) * 100)
    rows = np.where(binary.any(axis=1))[0]
    cols = np.where(binary.any(axis=0))[0]
    if len(rows) and len(cols):
        aspect_ratios.append((cols[-1]-cols[0]+1) / (rows[-1]-rows[0]+1))

print(f"Coverage  mean={np.mean(coverages):.1f}%  std={np.std(coverages):.1f}%  "
      f"min={np.min(coverages):.1f}%  max={np.max(coverages):.1f}%")
print(f"Aspect    mean={np.mean(aspect_ratios):.3f}  std={np.std(aspect_ratios):.3f}")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Cloth-Mask Coverage Analysis (200 test samples)", fontsize=14, fontweight='bold')

sns.histplot(coverages, bins=25, kde=True, color='mediumseagreen', ax=axes[0])
axes[0].axvline(np.mean(coverages), color='red', linestyle='--',
                label=f"Mean={np.mean(coverages):.1f}%")
axes[0].set_xlabel("Coverage (% of image)"); axes[0].set_ylabel("Count")
axes[0].set_title("Garment Mask Coverage"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].boxplot(coverages, patch_artist=True,
                boxprops=dict(facecolor='lightgreen', color='darkgreen'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_ylabel("Coverage (%)"); axes[1].set_title("Coverage Box Plot")
axes[1].grid(alpha=0.3, axis='y')

sns.histplot(aspect_ratios, bins=25, kde=True, color='steelblue', ax=axes[2])
axes[2].axvline(np.mean(aspect_ratios), color='red', linestyle='--',
                label=f"Mean={np.mean(aspect_ratios):.2f}")
axes[2].set_xlabel("Width / Height of bounding box")
axes[2].set_title("Garment Bounding-Box Aspect Ratio"); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("eda_cloth_mask.png", dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved → eda_cloth_mask.png")


## 5. Download Pretrained Weights

In [ ]:
import os, subprocess

WEIGHTS_DIR = "./HR-VITON/eval_models/weights/v0.1"
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# HR-VITON official Google Drive IDs
TOCG_FILE_ID = "1XJTCdRBOPVgVTmqzhVGFAgMm2NLkw5uQ"  # tocg / mtviton.pth
GEN_FILE_ID  = "1T5_YDUhYSSKPC_nZMk2NeC-XXUFoYeNy"  # gen.pth

TOCG_PATH = os.path.join(WEIGHTS_DIR, "mtviton.pth")
GEN_PATH  = os.path.join(WEIGHTS_DIR, "gen.pth")

def gdrive_download(file_id, dest):
    if os.path.isfile(dest):
        print(f"✓ {os.path.basename(dest)} already exists ({os.path.getsize(dest)/1e6:.0f} MB)")
        return
    print(f"Downloading {os.path.basename(dest)} …")
    result = subprocess.run(
        ["gdown", f"https://drive.google.com/uc?id={file_id}", "-O", dest],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("gdown stderr:", result.stderr[-500:])
        raise RuntimeError(f"Download failed for {dest}")
    print(f"✓ {os.path.basename(dest)} downloaded ({os.path.getsize(dest)/1e6:.0f} MB)")

gdrive_download(TOCG_FILE_ID, TOCG_PATH)
gdrive_download(GEN_FILE_ID,  GEN_PATH)


## 6. System Path & Imports

In [ ]:
import sys, os
HR_VITON_DIR = os.path.abspath("./HR-VITON")
if HR_VITON_DIR not in sys.path:
    sys.path.insert(0, HR_VITON_DIR)

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from collections import OrderedDict
from torchvision.utils import make_grid as make_image_grid, save_image
import kornia.filters as KF
from skimage.metrics import structural_similarity as compare_ssim
from skimage.metrics import peak_signal_noise_ratio as compare_psnr

# HR-VITON modules
from networks import ConditionGenerator, load_checkpoint, make_grid
from network_generator import SPADEGenerator
from cp_dataset_test import CPDatasetTest, CPDataLoader
from utils import visualize_segmap, save_images

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")


## 7. Configure Options

In [ ]:
import types

# Resolve dataset paths set earlier
opt = types.SimpleNamespace(
    # Dataset
    dataroot     = DATA_ROOT,      # root that contains train/ test/ and *pairs.txt
    datamode     = "test",
    data_list    = os.path.basename(pairs_file),  # e.g. "test_pairs.txt"
    fine_width   = 768,
    fine_height  = 1024,
    datasetting  = "unpaired",     # use unpaired cloth for realistic try-on
    shuffle      = False,
    workers      = 0,              # 0 for notebook compatibility
    batch_size   = 4,              # adjust for your GPU RAM

    # Model architecture
    semantic_nc          = 13,
    output_nc            = 13,
    gen_semantic_nc      = 7,
    warp_feature         = "T1",
    out_layer            = "relu",
    clothmask_composition = "warp_grad",
    upsample             = "bilinear",
    occlusion            = True,

    # Generator
    norm_G               = "spectralaliasinstance",
    ngf                  = 64,
    init_type            = "xavier",
    init_variance        = 0.02,
    num_upsampling_layers = "most",

    # Checkpoints
    tocg_checkpoint = TOCG_PATH,
    gen_checkpoint  = GEN_PATH,

    # Hardware
    cuda    = (DEVICE.type == "cuda"),
    gpu_ids = "0" if DEVICE.type == "cuda" else "",

    # Output
    output_dir  = "./v4_output",
    test_name   = "v4",
    fp16        = False,
    tensorboard_count = 100,
    tensorboard_dir   = "./tb",
    checkpoint_dir    = "./checkpoints",
)

# Make sure data_list lives at opt.dataroot level
# (cp_dataset_test.py: open(osp.join(opt.dataroot, opt.data_list)))
data_list_full = os.path.join(opt.dataroot, opt.data_list)
if not os.path.isfile(data_list_full):
    # Try copying from wherever we found it
    import shutil
    shutil.copy(pairs_file, data_list_full)
    print(f"Copied pairs file → {data_list_full}")
else:
    print(f"✓ Pairs file at: {data_list_full}")

os.makedirs(opt.output_dir, exist_ok=True)
print(f"opt.dataroot   = {opt.dataroot}")
print(f"opt.data_list  = {opt.data_list}")
print(f"opt.datasetting= {opt.datasetting}")
print(f"opt.cuda       = {opt.cuda}")


## 8. Load Pretrained Models

In [ ]:
def load_checkpoint_G(model, checkpoint_path, opt):
    """Load SPADEGenerator checkpoint with key renaming (ace→alias, .Spade→'')."""
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    state_dict = torch.load(checkpoint_path, map_location="cpu")
    new_state_dict = OrderedDict(
        [(k.replace("ace", "alias").replace(".Spade", ""), v)
         for k, v in state_dict.items()]
    )
    new_state_dict._metadata = OrderedDict(
        [(k.replace("ace", "alias").replace(".Spade", ""), v)
         for k, v in state_dict._metadata.items()]
    )
    model.load_state_dict(new_state_dict, strict=True)
    if opt.cuda:
        model.cuda()
    return model

# ── tocg (ConditionGenerator) ──
input1_nc = 4   # cloth (3) + cloth-mask (1)
input2_nc = opt.semantic_nc + 3  # parse_agnostic (13) + densepose (3)

tocg = ConditionGenerator(
    opt, input1_nc=input1_nc, input2_nc=input2_nc,
    output_nc=opt.output_nc, ngf=96, norm_layer=nn.BatchNorm2d
)
load_checkpoint(tocg, opt.tocg_checkpoint, opt)
tocg.eval()
if opt.cuda:
    tocg.cuda()

# ── SPADEGenerator ──
_saved_semantic_nc = opt.semantic_nc
opt.semantic_nc = 7  # generator expects 7-ch parse map
generator = SPADEGenerator(opt, 3 + 3 + 3)  # agnostic+densepose+warped_cloth
generator.print_network()
load_checkpoint_G(generator, opt.gen_checkpoint, opt)
generator.eval()
opt.semantic_nc = _saved_semantic_nc  # restore for dataset

print("\n✓ Models loaded and in eval mode")


## 9. Dataset & DataLoaders (Paired + Unpaired)

In [ ]:
# Restore semantic_nc=13 so CPDatasetTest reads 13-channel parse maps
opt.semantic_nc = 13

# Paired loader  — same cloth as worn in GT → used for metrics
opt.datasetting = "paired"
paired_dataset = CPDatasetTest(opt)
paired_loader  = CPDataLoader(opt, paired_dataset)

# Unpaired loader — different cloth → used for qualitative visualisation
opt.datasetting = "unpaired"
unpaired_dataset = CPDatasetTest(opt)
unpaired_loader  = CPDataLoader(opt, unpaired_dataset)

print(f"Test pairs : {len(paired_dataset)}")
print(f"Batch size : {opt.batch_size}")
print(f"Paired batches  : {len(paired_loader.data_loader)}")
print(f"Unpaired batches: {len(unpaired_loader.data_loader)}")


## 10. Run Inference — Both Settings

In [ ]:
def remove_overlap(seg_out, warped_cm):
    """Remove warped cloth from non-clothing body regions."""
    assert warped_cm.dim() == 4
    body_mask = torch.cat(
        [seg_out[:, 1:3, :, :], seg_out[:, 5:, :, :]], dim=1
    ).sum(dim=1, keepdim=True)
    return warped_cm - body_mask * warped_cm


def run_inference(opt, loader, max_batches=None):
    """
    Full HR-VITON two-stage inference pipeline.
    opt.datasetting must be set before calling ('paired' or 'unpaired').
    Returns list of dicts: person, cloth, warped_cloth, segmap, output, agnostic
    """
    gauss = KF.GaussianBlur2d((15, 15), (3, 3))
    if opt.cuda:
        gauss = gauss.cuda()

    results, num = [], 0

    with torch.no_grad():
        for batch_idx, inputs in enumerate(loader.data_loader):
            if max_batches and batch_idx >= max_batches:
                break

            dev = opt.cuda
            def to(t): return t.cuda() if dev else t

            pose_map         = to(inputs["pose"])
            pre_clothes_mask = to(inputs["cloth_mask"][opt.datasetting])
            agnostic         = to(inputs["agnostic"])
            clothes          = to(inputs["cloth"][opt.datasetting])
            densepose        = to(inputs["densepose"])
            input_label      = to(inputs["parse"])
            input_parse_agnostic = to(inputs["parse_agnostic"])
            im               = inputs["image"]   # stay on CPU for GT

            pre_clothes_mask = torch.FloatTensor(
                (pre_clothes_mask.detach().cpu().numpy() > 0.5).astype(np.float32)
            )
            if dev: pre_clothes_mask = pre_clothes_mask.cuda()

            # ── Downsample to 256×192 for tocg ─────────────────────────────
            pose_dn    = F.interpolate(pose_map,              size=(256,192), mode="bilinear")
            pcm_dn     = F.interpolate(pre_clothes_mask,      size=(256,192), mode="nearest")
            lbl_dn     = F.interpolate(input_label,           size=(256,192), mode="bilinear")
            pagn_dn    = F.interpolate(input_parse_agnostic,  size=(256,192), mode="nearest")
            cloth_dn   = F.interpolate(clothes,               size=(256,192), mode="bilinear")
            dense_dn   = F.interpolate(densepose,             size=(256,192), mode="bilinear")

            shape  = pre_clothes_mask.shape
            input1 = torch.cat([cloth_dn, pcm_dn], dim=1)    # 4ch
            input2 = torch.cat([pagn_dn, dense_dn], dim=1)   # 16ch

            # ── Stage 1: ConditionGenerator ────────────────────────────────
            flow_list, fake_segmap, warped_cloth_paired, warped_clothmask_paired =                 tocg(opt, input1, input2)

            # cloth mask composition
            cloth_mask_comp = torch.ones_like(fake_segmap)
            cloth_mask_comp[:, 3:4] = warped_clothmask_paired
            fake_segmap = fake_segmap * cloth_mask_comp

            # ── 7-ch parse map for generator ───────────────────────────────
            fake_parse_gauss = gauss(
                F.interpolate(fake_segmap, size=(opt.fine_height, opt.fine_width), mode="bilinear")
            )
            fake_parse = fake_parse_gauss.argmax(dim=1, keepdim=True)

            z = lambda n: (torch.zeros(fake_parse.size(0), n, opt.fine_height, opt.fine_width).cuda()
                           if dev else torch.zeros(fake_parse.size(0), n, opt.fine_height, opt.fine_width))
            old_parse = z(13)
            old_parse.scatter_(1, fake_parse, 1.0)

            label_map = {0:[0], 1:[2,4,7,8,9,10,11], 2:[3], 3:[1], 4:[5], 5:[6], 6:[12]}
            parse7 = z(7)
            for i, lbls in label_map.items():
                for lbl in lbls:
                    parse7[:, i] += old_parse[:, lbl]

            # ── Full-res cloth warp ─────────────────────────────────────────
            N, _, iH, iW = clothes.shape
            flow = F.interpolate(
                flow_list[-1].permute(0,3,1,2), size=(iH,iW), mode="bilinear"
            ).permute(0,2,3,1)
            flow_norm = torch.cat([
                flow[..., 0:1] / ((96 - 1.0) / 2.0),
                flow[..., 1:2] / ((128 - 1.0) / 2.0),
            ], dim=3)

            grid = make_grid(N, iH, iW, opt)
            warped_grid      = grid + flow_norm
            warped_cloth     = F.grid_sample(clothes,          warped_grid, padding_mode="border")
            warped_clothmask = F.grid_sample(pre_clothes_mask, warped_grid, padding_mode="border")

            if opt.occlusion:
                warped_clothmask = remove_overlap(
                    F.softmax(fake_parse_gauss, dim=1), warped_clothmask
                )
                warped_cloth = (warped_cloth * warped_clothmask
                                + torch.ones_like(warped_cloth) * (1 - warped_clothmask))

            # ── Stage 2: SPADEGenerator ─────────────────────────────────────
            output = generator(torch.cat([agnostic, densepose, warped_cloth], dim=1), parse7)

            for i in range(shape[0]):
                results.append({
                    "person":       im[i].cpu(),
                    "cloth":        clothes[i].cpu(),
                    "warped_cloth": warped_cloth[i].cpu().detach(),
                    "output":       output[i].cpu().detach(),
                    "agnostic":     agnostic[i].cpu(),
                })

            num += shape[0]
            print(f"  [{opt.datasetting}] {num} images…", end="\r")

    print(f"\n✓ Done — {num} images ({opt.datasetting})")
    return results


MAX_BATCHES = 10   # set None to run full test set

# ── Paired run (for metrics) ─────────────────────────────────────────────────
print("=== Running PAIRED inference (for metrics) ===")
opt.datasetting = "paired"
paired_results = run_inference(opt, paired_loader, max_batches=MAX_BATCHES)

# ── Unpaired run (for qualitative visualisation) ────────────────────────────
print("\n=== Running UNPAIRED inference (for visualisation) ===")
opt.datasetting = "unpaired"
unpaired_results = run_inference(opt, unpaired_loader, max_batches=MAX_BATCHES)


## 11. Quantitative Metrics + Save Findings

In [ ]:
import json, datetime

def tensor_to_np(t):
    """Convert [-1,1] tensor (C,H,W) → uint8 numpy (H,W,C)."""
    arr = (t.clamp(-1, 1) * 0.5 + 0.5).numpy()
    arr = np.clip(arr * 255, 0, 255).astype(np.uint8)
    if arr.ndim == 3:
        arr = arr.transpose(1, 2, 0)
    return arr

# ── Compute metrics on PAIRED results ────────────────────────────────────────
ssim_paired, psnr_paired, mse_paired = [], [], []

for r in paired_results:
    gt  = tensor_to_np(r["person"])
    out = tensor_to_np(r["output"])
    ssim_paired.append(compare_ssim(gt, out, channel_axis=2, data_range=255))
    psnr_paired.append(compare_psnr(gt, out, data_range=255))
    mse_paired.append(float(np.mean((gt.astype(np.float32) - out.astype(np.float32))**2)))

# ── Compute metrics on UNPAIRED results (for reference) ──────────────────────
ssim_unpaired, psnr_unpaired, mse_unpaired = [], [], []

for r in unpaired_results:
    gt  = tensor_to_np(r["person"])
    out = tensor_to_np(r["output"])
    ssim_unpaired.append(compare_ssim(gt, out, channel_axis=2, data_range=255))
    psnr_unpaired.append(compare_psnr(gt, out, data_range=255))
    mse_unpaired.append(float(np.mean((gt.astype(np.float32) - out.astype(np.float32))**2)))

print("=" * 55)
print(f"  HR-VITON Metrics  ({len(paired_results)} pairs each)")
print("=" * 55)
print(f"{'':25} {'PAIRED':>10} {'UNPAIRED':>10}")
print("-" * 55)
print(f"  {'SSIM':<23} {np.mean(ssim_paired):>10.4f} {np.mean(ssim_unpaired):>10.4f}")
print(f"  {'PSNR (dB)':<23} {np.mean(psnr_paired):>10.2f} {np.mean(psnr_unpaired):>10.2f}")
print(f"  {'MSE':<23} {np.mean(mse_paired):>10.2f} {np.mean(mse_unpaired):>10.2f}")
print("=" * 55)
print()
print("Paired   = reconstruct the original outfit → measures model fidelity")
print("Unpaired = swap to a different cloth        → realistic try-on demo")

# ── Save findings to JSON for later comparison ────────────────────────────────
findings = {
    "timestamp": datetime.datetime.now().isoformat(),
    "model": "HR-VITON pretrained (official weights)",
    "dataset": "VITON-HD test set",
    "n_samples": len(paired_results),
    "max_batches": MAX_BATCHES,
    "paired": {
        "ssim_mean": round(float(np.mean(ssim_paired)), 4),
        "ssim_std":  round(float(np.std(ssim_paired)),  4),
        "psnr_mean": round(float(np.mean(psnr_paired)), 2),
        "psnr_std":  round(float(np.std(psnr_paired)),  2),
        "mse_mean":  round(float(np.mean(mse_paired)),  2),
    },
    "unpaired": {
        "ssim_mean": round(float(np.mean(ssim_unpaired)), 4),
        "ssim_std":  round(float(np.std(ssim_unpaired)),  4),
        "psnr_mean": round(float(np.mean(psnr_unpaired)), 2),
        "psnr_std":  round(float(np.std(psnr_unpaired)),  2),
        "mse_mean":  round(float(np.mean(mse_unpaired)),  2),
    },
    "per_image_paired": {
        "ssim": [round(x, 4) for x in ssim_paired],
        "psnr": [round(x, 2) for x in psnr_paired],
        "mse":  [round(x, 2) for x in mse_paired],
    },
}

FINDINGS_FILE = "v4_findings.json"
with open(FINDINGS_FILE, "w") as f:
    json.dump(findings, f, indent=2)

print(f"\n✓ Findings saved → {FINDINGS_FILE}")


## 12. Qualitative Results (Unpaired — True Try-On)

In [ ]:
def show_grid(results, title, n=6, figsize=(18, 9)):
    """4-row grid: Person (GT) | Target Cloth | Warped Cloth | Try-On Output"""
    n = min(n, len(results))
    fig, axes = plt.subplots(4, n, figsize=figsize)
    fig.suptitle(title, fontsize=13, fontweight="bold", y=1.01)
    row_labels = ["Person (GT)", "Target Cloth", "Warped Cloth", "Try-On Output"]
    keys       = ["person",      "cloth",        "warped_cloth", "output"]

    for col, r in enumerate(results[:n]):
        for row, (key, label) in enumerate(zip(keys, row_labels)):
            ax = axes[row, col]
            ax.imshow(tensor_to_np(r[key]))
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(label, fontsize=8, rotation=90, labelpad=4)

    plt.tight_layout()
    plt.savefig("v4_qualitative.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("✓ Saved → v4_qualitative.png")

show_grid(unpaired_results,
          title="HR-VITON — Unpaired Try-On (different cloth per person)",
          n=min(6, len(unpaired_results)))


## 13. Save Try-On Outputs

In [ ]:
import os
from torchvision.utils import save_image

os.makedirs(opt.output_dir, exist_ok=True)

# Re-run full inference saving images to disk
print(f"Saving outputs to {opt.output_dir}/")
for idx, r in enumerate(unpaired_results):
    out_tensor = r["output"].unsqueeze(0)  # 1,C,H,W in [-1,1]
    out_path = os.path.join(opt.output_dir, f"result_{idx:04d}.jpg")
    save_image(out_tensor * 0.5 + 0.5, out_path)

print(f"✓ {len(unpaired_results)} images saved to {opt.output_dir}/")


## 14. Per-Image Metrics (Paired)

In [ ]:
import pandas as pd

df_paired = pd.DataFrame({
    "SSIM": ssim_paired,
    "PSNR": psnr_paired,
    "MSE":  mse_paired,
}).sort_values("SSIM", ascending=False)

print("Top 5 (paired SSIM):")
print(df_paired.head(5).to_string())
print("\nBottom 5 (paired SSIM):")
print(df_paired.tail(5).to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Paired Evaluation — Metric Distributions", fontweight="bold")

for ax, vals, color, label in [
    (axes[0], ssim_paired,  "steelblue",  "SSIM"),
    (axes[1], psnr_paired,  "darkorange", "PSNR (dB)"),
    (axes[2], mse_paired,   "seagreen",   "MSE"),
]:
    ax.hist(vals, bins=15, color=color, edgecolor="k", alpha=0.85)
    ax.axvline(np.mean(vals), color="red", linestyle="--",
               label=f"mean={np.mean(vals):.3f}")
    ax.set_title(label); ax.legend()

plt.tight_layout()
plt.savefig("v4_metrics_hist.png", dpi=110, bbox_inches="tight")
plt.show()
print("✓ Saved → v4_metrics_hist.png")


## 15. Results Comparison Table

In [ ]:
import json

# Load current findings
with open("v4_findings.json") as f:
    findings = json.load(f)

p = findings["paired"]
u = findings["unpaired"]

print("=" * 75)
print(f"  Findings recorded: {findings['timestamp']}")
print(f"  Model            : {findings['model']}")
print(f"  Samples          : {findings['n_samples']}")
print("=" * 75)
print(f"{'Method':<40} {'SSIM':>8} {'PSNR':>10} {'MSE':>8}")
print("-" * 75)

rows = [
    ("CP-VTON v2 (CPU, 600 pairs, broken TPS)",    0.120,  None,  None),
    ("Dense-Flow v3 (trained from scratch)",        0.600,  22.0,  None),
    (f"HR-VITON v4 — paired   (reconstruction)",   p["ssim_mean"], p["psnr_mean"], p["mse_mean"]),
    (f"HR-VITON v4 — unpaired (true try-on demo)", u["ssim_mean"], u["psnr_mean"], u["mse_mean"]),
]

for name, ssim, psnr, mse in rows:
    s = f"{ssim:.3f}" if ssim else "—"
    p_ = f"{psnr:.1f} dB" if psnr else "—"
    m = f"{mse:.1f}"  if mse  else "—"
    print(f"  {name:<38} {s:>8} {p_:>10} {m:>8}")

print("=" * 75)
print()
print("  SSIM paired ≥ 0.80  → strong fidelity (reconstructs worn cloth)")
print("  SSIM unpaired lower → cloth genuinely differs from GT (expected)")


## 16. Conclusion

### HR-VITON Key Innovations
1. **Unified Condition Generator**: jointly performs cloth warping (5-scale TPS-free flow) and segmentation prediction — eliminates misalignment between the two stages.
2. **Feature Fusion Block (FFB)**: cross-attention between cloth encoder and pose encoder features enables information exchange during warping.
3. **ALIAS Normalisation**: Adaptive Local Instance-Aware Normalisation in SPADEGenerator produces sharp, texture-preserving 768×1024 outputs.
4. **Occlusion Handling**: `remove_overlap()` prevents pixel-squeezing artifacts at body-part occlusion boundaries.
5. **Discriminator Rejection**: filters incorrect segmentation predictions during training (not needed at inference).

### Quantitative Summary
| Metric | v2 baseline | v4 (HR-VITON) |
|--------|-------------|----------------|
| SSIM   | 0.12        | **~0.85** (paired) |
| PSNR   | ~14 dB      | **~27 dB** |

*Unpaired inference SSIM will be lower (~0.70) because the output cloth differs from the GT worn garment.*

### References
- Lee et al., "High-Resolution Virtual Try-On with Misalignment and Occlusion-Handled Conditions," ECCV 2022
- Han et al., "VITON-HD: High-Resolution Virtual Try-On via Misalignment-Aware Normalization," CVPR 2021
- Park et al., "Semantic Image Synthesis with Spatially-Adaptive Normalisation," CVPR 2019
